In [1]:
import base64
import requests
import os
import json
from PIL import Image


api_base = "https://neudm.zeabur.app/v1"
api_key = "实际key"
# image_path = "data/images/normal"
# json_path = "normal"
# output_path = "data/res_summary_len8"
image_path = "/Users/leon/Desktop/topo2text/data/test_data/normal"
json_path = "/Users/leon/Desktop/topo2text/data/result/test_data/normal"
output_path = "data/res_summary_test"
# with open('data/result/normal/n1058.json', 'r') as file:
#     QA_pairs = json.load(file)
if not os.path.exists(output_path):
    os.makedirs(output_path)

In [2]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def get_image_summary(base64_image, json_data=None):

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}",
    }

    # 删除QA pairs相关内容，测试发现，其实我们并不需要QA pair来帮助他总结，此处QA pair是作为额外的训练数据的
    # TODO 丰富4内容，增加更多信息，以及如何处理QA pairs
    # 写成 ： 这个summary要以这样的形式：1.先两个句子概括整个拓扑图的结构和大致内容，2. 3-4个句子介绍拓扑图的细节. 3. 两句句子描述拓扑图的功能
    content_system = (
        "You are a professional network topology analysis expert."
        "Your goal is to accurately and clearly analyze and summarize a topology image\n"
        "You will be provided two information:"
        "1. A topology image "
        "2. (Optional) QA pairs related to the main elements of the topology. \n"
        "Please follow these steps: "
        "1. Image Analysis: Examine the topology image closely, identifying key nodes and their connections. "
        "2. Description Method Selection:** Choose a **layered description** (suitable for topologies with clear hierarchies, such as core and access layers) or a **distributed description** (suitable for topologies with scattered nodes) based on the observed structure."
        "3. Generate Summary: Using the chosen description method, create a summary of the topology. "
        "Output only an English summary of the topology image without any further analysis. The format should be: "
        "Output format: This network employs a [Topology type, e.g., ‘star,’ ‘ring,’ ‘tree,’ or ‘mesh’] architecture, primarily designed to [State the primary purpose of the network in one sentence, e.g., ‘facilitate internal communication,’ ‘provide secure remote access,’ ‘host web applications’]. Its core nodes include: [List core nodes, e.g., ‘central switches, routers, application servers, database servers’], interconnected and collaborating closely. For instance, [Node 1, e.g., ‘the front-end server’] interacts with [Node 2, e.g., ‘the load balancer’] via [Connection method, e.g., ‘HTTP requests’], responsible for [Node 1’s function, e.g., ‘handling incoming requests’]; [Node 2, e.g., ‘the load balancer’] forwards requests to [Node 3, e.g., ‘the application server’] through [Connection method, e.g., ‘TCP connections’], enabling [Node 2’s function, e.g., ‘load distribution’]; [Node 3, e.g., ‘the application server’] retrieves data from [Node 4, e.g., ‘the database server’] using [Connection method, e.g., ‘database connections’], fulfilling [Node 3’s function, e.g., ‘data processing’]; and [Node 5, e.g., ‘the monitoring server’] gathers status information via [Connection method, e.g., ‘SNMP protocol’], enabling [Node 5’s function, e.g., ‘system monitoring’]. These nodes are linked through [Specify the dependencies between nodes, e.g., ‘data dependencies, control dependencies, and collaborative dependencies’]. Overall, this topology aims to [Summarize the overall purpose of the network topology, emphasizing how it meets network goals, e.g., ‘improve system availability,’ ‘ensure secure deployment,’ ‘flexibly accommodate future growth,’ ‘guarantee reliable information flow,’ ‘provide a stable service’]."
    )

    messages = [
        {"role": "system", "content": content_system},
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"<Topo_image>, QA pairs: {json_data}.",
                },
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"},
                },
            ],
        },
    ]

    payload = {
        "model": "gpt-4o-mini",
        "messages": messages,
        "temperature": 0.2,
        # "max_tokens": 300,
    }
    try:
        response = requests.post(
            f"{api_base}/chat/completions", headers=headers, json=payload
        )
        response.raise_for_status()  
        result = response.json()
        summary = result["choices"][0]["message"]["content"]
        return summary
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        print(f"Response content: {response.text}")
        return None
    except (KeyError, IndexError, json.JSONDecodeError) as e:
        print(f"Error processing response: {e}")
        return None


# 遍历图片目录并处理每个图片
for filename in os.listdir(image_path):
    if filename.lower().endswith((".png", ".jpg", ".jpeg", ".gif", ".bmp", ".webp")):
        image_filepath = os.path.join(image_path, filename)
        json_filename = os.path.splitext(filename)[0] + ".json"
        json_filepath = os.path.join(json_path, json_filename)

        # 读取JSON文件内容（如果存在）
        json_data = None
        if os.path.exists(json_filepath):
            try:
                with open(json_filepath, "r", encoding="utf-8") as json_file:
                    json_data = json.load(json_file)
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON from {json_filepath}: {e}")
                continue

        # 编码图片并获取总结
        base64_image = encode_image(image_filepath)
        if base64_image:
            summary = get_image_summary(base64_image, json_data)
            if summary:
                txt_filename = os.path.splitext(filename)[0] + ".txt"
                txt_filepath = os.path.join(output_path, txt_filename)
                try:
                    with open(txt_filepath, "w", encoding="utf-8") as txt_file:
                        txt_file.write(summary)
                    print(f"Summary for {filename} saved to {txt_filename}")
                except Exception as e:
                    print(f"Error saving summary for {filename}: {e}")
        else:
            print(f"Skipping invalid or unsupported image file: {filename}")
    else:
        print(f"Skipping non-image file: {filename}")

Summary for n1059.png saved to n1059.txt
Summary for n1058.png saved to n1058.txt
Summary for n1062.png saved to n1062.txt
Summary for n1060.jpg saved to n1060.txt
Summary for n1061.jpg saved to n1061.txt
